In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

In [ ]:
# cellule 1 — se placer dans le dossier (magie IPython, seule sur sa ligne)
%cd /content/grobid-0.8.2

In [ ]:
# cellule 2 — compiler (5 à 10 minutes)
# !./gradlew clean install --no-daemon -q

In [ ]:
!wget -q https://github.com/kermitt2/grobid/archive/0.8.2.zip
!unzip -qq 0.8.2.zip
%cd grobid-0.8.2
!./gradlew clean install --no-daemon -q

In [ ]:
# cellule 3 — démarrer le serveur en arrière-plan
import subprocess, os
os.chdir("/content/grobid-0.8.2")
subprocess.Popen(["./gradlew", "run", "--no-daemon"],
                 stdout=open("/content/grobid.log", "w"),
                 stderr=subprocess.STDOUT)
os.chdir("/content")

In [ ]:
# cellule 4 — attendre et vérifier (le démarrage prend 1 à 3 minutes)
import requests, time
for i in range(40):
    try:
        r = requests.get("http://localhost:8070/api/isalive", timeout=5)
        if r.status_code == 200 and r.text.strip().lower() == "true":
            print("GROBID actif ✓"); break
    except Exception:
        pass
    time.sleep(10)
else:
    print("pas encore prêt — regardez /content/grobid.log")
    print(open("/content/grobid.log").read()[-2000:])

In [ ]:
!pip install requests pandas pyarrow lxml beautifulsoup4 pymupdf openpyxl -q

In [ ]:
#!/usr/bin/env python3
"""
fulltext_harvester_v5.py  —  Google Colab
=========================================
Récolte de texte intégral : voie QUALITÉ d'abord (JATS structuré avec vraies
balises de section), puis voie COUVERTURE (Unpaywall live + routage par éditeur
+ découverte du lien PDF dans le HTML), PDF -> GROBID.

Couches, dans l'ordre, par étude :
  1. Europe PMC fullTextXML      JATS  (meilleure structure)
  2. PMC E-utilities efetch      JATS
  3. bioRxiv / medRxiv           JATS
  4. arXiv                       GROBID/résumé
  5. Unpaywall LIVE -> candidats (url_for_pdf + landing page)
  6. routage par famille d'éditeur (landing-first vs pdf-first)
  7. HTML -> scrape citation_pdf_url -> récupère le vrai PDF
  8. PDF -> GROBID si dispo, sinon découpage regex

== NOUVEAUTÉS v5 (d'après le diagnostic de couverture) ==
  1. ÉCHELLE D'IDENTIFIANTS dans try_epmc : PMCID direct -> DOI -> PMID ->
     convertisseur NCBI -> TITRE. Le fichier d'entrée contenait 0 PMCID et
     19 012 PMID ; ~7 800 PMCID étaient récupérables mais jamais cherchés.
  2. try_pmc : repli convertisseur PMID quand oa_pmcid est vide.
  3. BUG CORRIGÉ : discover_pdf_links contenait "fulltext3" (remplacement
     accidentel lors du renommage du dossier) -> cassait la découverte PDF.
  4. NCBI_KEY lu depuis les secrets Colab (3 -> 10 req/s).
  5. ÉCRITURES RÉSILIENTES : Drive monté échoue sur de gros volumes de petits
     fichiers (OSError Errno 5). Écriture locale + synchronisation Drive par
     lots, avec reprise.
  6. Secrets lus UNE SEULE FOIS au démarrage (évite le TimeoutException Colab
     au milieu d'une exécution de plusieurs heures).
  7. RANGE_START / RANGE_END explicites au lieu d'un iloc codé en dur.
  8. Journal des causes d'échec enrichi (dernière étape = cause réelle).

SETUP
  # GROBID (recommandé) :
  !apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
  import os; os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
  !wget -q https://github.com/kermitt2/grobid/archive/0.8.2.zip && unzip -qq 0.8.2.zip
  %cd grobid-0.8.2 && !./gradlew clean install --no-daemon -q
  import subprocess; subprocess.Popen(["./gradlew","run","--no-daemon"],
      stdout=open("/content/grobid.log","w"), stderr=subprocess.STDOUT)
  %cd /content
  !pip install requests pandas pyarrow lxml beautifulsoup4 pymupdf openpyxl -q
"""

import io, json, os, re, shutil, sys, time
from pathlib import Path
from urllib.parse import urlparse, urljoin
import requests
import pandas as pd
from lxml import etree

# --------------------------------------------------------------------------- #
# secrets lus UNE SEULE FOIS (v5-6) — évite le TimeoutException à mi-parcours
# --------------------------------------------------------------------------- #
def _secret(name, default=""):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    return os.environ.get(name, default)


# =========================================================================== #
# CONFIG                                                                      #
# =========================================================================== #
IN_FILE     = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/merged_human_class2_pmcid.xlsx"
INCLUDE_COL, INCLUDE_VAL = "last_final_include", "include"

DRIVE_OUT   = "/content/drive/MyDrive/COGNOSX/ticks/fulltext3"
LOCAL_OUT   = "/content/_ft"                 # espace de travail local (v5-5)

RANGE_START = 0            # v5-7 : tranche explicite ; 0/None = tout le corpus
RANGE_END   = None

MAILTO      = "cognosxx@gmail.com"
NCBI_KEY    = _secret("NCBI_KEY")            # v5-4 : gratuit, 3 -> 10 req/s

GROBID_URL  = "http://localhost:8070"        # "" = découpage regex des PDF
GROBID_TIMEOUT = 180

# True  = re-traite les succès plain-text pour tenter de les upgrader (phase QUALITÉ)
# False = ne comble que les vrais manques (phase COUVERTURE) <- à utiliser d'abord
REQUIRE_STRUCTURED_FOR_FINISHED = False

MIN_BODY_CHARS = 200
SLEEP        = 0.7          # politesse entre requêtes web uniquement
TIMEOUT      = 45
SYNC_EVERY   = 200          # études entre deux synchronisations Drive

# --- chemins dérivés -------------------------------------------------------
DRIVE_RAW, DRIVE_SEC = DRIVE_OUT + "/raw", DRIVE_OUT + "/sections"
LOCAL_RAW, LOCAL_SEC = LOCAL_OUT + "/raw", LOCAL_OUT + "/sections"
MANIFEST   = DRIVE_OUT + "/fulltext_manifest.parquet"
LOCAL_MAN  = LOCAL_OUT + "/fulltext_manifest.parquet"

sys.path.insert(0, "/content/drive/MyDrive/Colab Notebooks")
sys.path.insert(0, "/content")
from tick_sections import (classify_line, classify_heading, parse_plaintext,
                           has_targets as _has_targets_shared, TARGET_SECTIONS)

GROBID_URL = GROBID_URL.rstrip("/")
for d in (DRIVE_RAW, DRIVE_SEC, LOCAL_RAW, LOCAL_SEC):
    Path(d).mkdir(parents=True, exist_ok=True)

SESS = requests.Session()
SESS.headers.update({"User-Agent": f"tick-sos-fulltext/5.0 (mailto:{MAILTO})",
                     "Accept": "text/html,application/xml,application/pdf,*/*"})


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def short_id(x): return S(x).rsplit("/", 1)[-1]
def norm_doi(x): return S(x).lower().replace("https://doi.org/", "").replace("doi.org/", "").strip()
def clean_pmid(x): return re.sub(r"\D", "", S(x))


# =========================================================================== #
# garde de santé GROBID                                                       #
# =========================================================================== #
def check_grobid(url, wait=0):
    if not url: return False
    deadline = time.time() + wait
    while True:
        try:
            r = requests.get(f"{url}/api/isalive", timeout=10)
            if r.status_code == 200 and r.text.strip().lower() == "true":
                return True
        except requests.RequestException:
            pass
        if time.time() >= deadline: return False
        time.sleep(5)

if GROBID_URL:
    if not check_grobid(GROBID_URL, wait=60):
        raise RuntimeError(
            f"GROBID_URL = {GROBID_URL} mais le serveur ne répond pas.\n"
            f"Démarrez-le (voir SETUP, puis /content/grobid.log), ou mettez "
            f"GROBID_URL = '' pour accepter le découpage regex.")
    print(f"[grobid] actif sur {GROBID_URL}", file=sys.stderr)
else:
    print("[grobid] désactivé — les PDF seront découpés par regex", file=sys.stderr)
print(f"[ncbi] clé API {'présente (10 req/s)' if NCBI_KEY else 'ABSENTE (3 req/s)'}",
      file=sys.stderr)


# =========================================================================== #
# GET robuste                                                                 #
# =========================================================================== #
def robust_get(url, expect="auto", params=None, tries=4):
    for attempt in range(tries):
        try:
            try:
                r = SESS.get(url, params=params, timeout=TIMEOUT, allow_redirects=True)
            except requests.exceptions.SSLError:
                r = SESS.get(url, params=params, timeout=TIMEOUT,
                             allow_redirects=True, verify=False)
            sc = r.status_code
            if sc == 200:
                ct = (r.headers.get("content-type") or "").lower()
                body = r.content
                kind = ("pdf" if body[:5].startswith(b"%PDF") or "pdf" in ct else
                        "html" if "html" in ct or b"<html" in body[:1000].lower() else
                        "xml" if "xml" in ct or body[:100].lstrip().startswith(b"<?xml") else
                        "other")
                if expect == "pdf" and kind != "pdf":
                    return "badformat", None, str(r.url), kind
                return "ok", (body if kind == "pdf" else r.text), str(r.url), kind
            if sc == 404: return "notfound", None, url, ""
            if sc == 410: return "gone", None, url, ""
            if sc in (401, 403): return "denied", None, url, ""
            if sc in (429, 500, 502, 503, 504):
                time.sleep(min(2 ** attempt, 30)); continue
            return "error", None, url, ""
        except requests.exceptions.RequestException:
            time.sleep(2 ** attempt)
    return "error", None, url, ""


# =========================================================================== #
# parseurs JATS / TEI                                                         #
# =========================================================================== #
SEC_TYPE_MAP = {"intro": "introduction", "introduction": "introduction",
    "methods": "methods", "materials|methods": "methods", "results": "results",
    "discussion": "discussion", "conclusion": "conclusion",
    "data-availability": "data_availability", "ethics-statement": "ethics",
    "supplementary-material": "supplementary", "software-availability": "software"}
TEI_BACK_MAP = {"availability": "data_availability", "funding": "funding",
                "acknowledgement": "acknowledgements", "acknowledgment": "acknowledgements",
                "annex": "supplementary", "appendix": "supplementary"}

def _txt(el):
    return re.sub(r"\s+", " ", "".join(el.itertext())).strip() if el is not None else ""

def parse_jats(xml_text):
    try:
        xml_text = re.sub(r"<!DOCTYPE[^>]*>", "", xml_text)
        root = etree.fromstring(xml_text.encode("utf-8"),
                                parser=etree.XMLParser(recover=True, resolve_entities=False))
    except Exception:
        return None
    out = {"sections": [], "refs": [],
           "language": root.get("{http://www.w3.org/XML/1998/namespace}lang") or ""}
    for p, lab in [(".//article-title", "title"), (".//abstract", "abstract")]:
        el = root.xpath(p)
        if el: out["sections"].append({"type": lab, "title": lab, "text": _txt(el[0])})
    for sec in root.xpath(".//body//sec"):
        stype = (sec.get("sec-type") or "").lower()
        title = _txt(sec.find("title"))
        label = SEC_TYPE_MAP.get(stype) or classify_line(title)[0] \
                or classify_heading(title) or (stype or "section")
        text = " ".join(x for x in (_txt(p) for p in sec.findall("p")) if x)
        if text: out["sections"].append({"type": label, "title": title, "text": text})
    for sec in (root.xpath(".//back//sec") + root.xpath(".//back//ack")
                + root.xpath(".//back//notes")):
        stype = (sec.get("sec-type") or "").lower()
        title = _txt(sec.find("title"))
        label = SEC_TYPE_MAP.get(stype) or classify_line(title)[0] \
                or classify_heading(title) or "back"
        text = _txt(sec)
        if text and len(text) > 5:
            out["sections"].append({"type": label, "title": title, "text": text})
    for ref in root.xpath(".//ref-list//ref"):
        doi = ""
        for pid in ref.xpath(".//pub-id"):
            if (pid.get("pub-id-type") or "") == "doi": doi = _txt(pid)
        out["refs"].append({"text": _txt(ref), "doi": doi})
    body = sum(len(s["text"]) for s in out["sections"]
               if s["type"] not in ("title", "abstract"))
    return out if body >= MIN_BODY_CHARS else None

def parse_tei(tei_text):
    """TEI GROBID -> sections. Lit <body> ET <back> (availability, funding,
    acknowledgements, annexes) : le back matter contient les sections cibles."""
    try:
        tei_text = re.sub(r"<\?xml[^>]*\?>", "", tei_text)
        root = etree.fromstring(tei_text.encode("utf-8"),
                                parser=etree.XMLParser(recover=True))
    except Exception:
        return None
    ns = {"t": "http://www.tei-c.org/ns/1.0"}
    out = {"sections": [], "refs": [],
           "language": root.get("{http://www.w3.org/XML/1998/namespace}lang") or ""}
    t = root.xpath("//t:titleStmt/t:title", namespaces=ns)
    if t: out["sections"].append({"type": "title", "title": "title", "text": _txt(t[0])})
    ab = root.xpath("//t:profileDesc//t:abstract", namespaces=ns)
    if ab: out["sections"].append({"type": "abstract", "title": "abstract", "text": _txt(ab[0])})
    for div in root.xpath("//t:body/t:div", namespaces=ns):
        head = div.xpath("t:head", namespaces=ns)
        title = _txt(head[0]) if head else ""
        text = " ".join(_txt(p) for p in div.xpath("t:p", namespaces=ns))
        if text:
            out["sections"].append({"type": classify_line(title)[0]
                                    or classify_heading(title) or "section",
                                    "title": title, "text": text})
    for div in root.xpath("//t:back//t:div", namespaces=ns):
        dtype = (div.get("type") or "").lower()
        head = div.xpath("t:head", namespaces=ns)
        title = _txt(head[0]) if head else ""
        text = _txt(div)
        if not text or len(text) <= 5: continue
        label = (TEI_BACK_MAP.get(dtype) or classify_line(title)[0]
                 or classify_line(text[:200])[0] or classify_heading(title)
                 or (dtype or "back"))
        out["sections"].append({"type": label, "title": title, "text": text})
    for bib in root.xpath("//t:listBibl/t:biblStruct", namespaces=ns):
        doi = ""
        d = bib.xpath(".//t:idno[@type='DOI']", namespaces=ns)
        if d: doi = norm_doi(_txt(d[0]))
        out["refs"].append({"text": _txt(bib), "doi": doi})
    body = sum(len(s["text"]) for s in out["sections"]
               if s["type"] not in ("title", "abstract"))
    return out if body >= MIN_BODY_CHARS else None


# =========================================================================== #
# extraction PDF / HTML                                                       #
# =========================================================================== #
def pdf_to_text(pdf_bytes):
    try:
        import fitz
        fitz.TOOLS.mupdf_display_errors(False)
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        return "\n".join(p.get_text() for p in doc)
    except Exception:
        return ""

def grobid_parse(pdf_bytes):
    if not GROBID_URL: return None
    try:
        r = SESS.post(f"{GROBID_URL}/api/processFulltextDocument",
                      files={"input": ("a.pdf", pdf_bytes, "application/pdf")},
                      data={"consolidateHeader": "0", "consolidateCitations": "0",
                            "segmentSentences": "0"}, timeout=GROBID_TIMEOUT)
        if r.status_code == 200: return r.text
        if r.status_code == 503:
            time.sleep(5)
            r = SESS.post(f"{GROBID_URL}/api/processFulltextDocument",
                          files={"input": ("a.pdf", pdf_bytes, "application/pdf")},
                          timeout=GROBID_TIMEOUT)
            return r.text if r.status_code == 200 else None
        return None
    except requests.RequestException:
        return None

def pdf_to_sections(pdf_bytes):
    tei = grobid_parse(pdf_bytes)
    if tei:
        parsed = parse_tei(tei)
        if parsed: return parsed, "grobid"
    txt = pdf_to_text(pdf_bytes)
    if txt:
        parsed = parse_plaintext(txt)
        if parsed: return parsed, "pymupdf_regex"
    return None, "pdf_unparsed"

def html_to_sections(html):
    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer"]): t.decompose()
        art = (soup.find("article")
               or soup.find("div", {"class": re.compile("article|fulltext|content")})
               or soup)
        return parse_plaintext(re.sub(r"\n{3,}", "\n\n", art.get_text("\n")))
    except Exception:
        return None


# =========================================================================== #
# routage éditeur + découverte du lien PDF                                    #
# =========================================================================== #
def domain_family(url):
    d = urlparse(S(url)).netloc.lower().replace("www.", "")
    for key, fam in [("ncbi.nlm.nih.gov", "pmc"), ("europepmc.org", "pmc"),
        ("mdpi.com", "mdpi"), ("pubs.acs.org", "acs"), ("onlinelibrary.wiley.com", "wiley"),
        ("academic.oup.com", "oup"), ("tandfonline.com", "tandf"), ("arxiv.org", "arxiv"),
        ("ssrn.com", "ssrn"), ("zenodo.org", "zenodo"), ("osf.io", "osf"),
        ("chemrxiv.org", "chemrxiv"), ("biomedcentral.com", "bmc"), ("plos.org", "plos"),
        ("frontiersin.org", "frontiers"), ("nature.com", "nature"),
        ("springer.com", "springer"), ("sciencedirect.com", "elsevier"),
        ("scielo", "scielo"), ("doiserbia", "generic")]:
        if key in d: return fam
    return "generic"

LANDING_FIRST = {"pmc", "ssrn", "wiley", "oup", "tandf", "mdpi", "acs", "nature", "elsevier"}

def rewrite_url(url):
    fam = domain_family(url); url = S(url).strip()
    if fam == "arxiv":
        url = re.sub(r"/abs/([^/?#]+)", r"/pdf/\1.pdf", url)
    return url

def oa_candidates(pdf_url, landing_url):
    fam = domain_family(pdf_url or landing_url)
    cands = []
    if fam in LANDING_FIRST:
        if landing_url: cands.append(("landing", landing_url))
        if pdf_url and pdf_url != landing_url: cands.append(("pdf", pdf_url))
    else:
        if pdf_url: cands.append(("pdf", pdf_url))
        if landing_url and landing_url != pdf_url: cands.append(("landing", landing_url))
    seen, out = set(), []
    for lab, u in cands:
        if u and u not in seen: seen.add(u); out.append((lab, u))
    return out

def discover_pdf_links(html, base_url):
    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(html, "html.parser")
        out = []
        for attrs in [{"name": "citation_pdf_url"}, {"name": "wkhealth_pdf_url"},
                      {"property": "og:pdf"}]:
            tag = soup.find("meta", attrs=attrs)
            if tag and tag.get("content"): out.append(urljoin(base_url, tag["content"]))
        # v5-3 BUG CORRIGÉ : "fulltext3" -> "fulltext" (+ variantes utiles)
        pats = [".pdf", "/pdf/", "article/download", "fulltext", "full-text",
                "pdfdirect", "/article-pdf/", "downloadpdf", "viewcontent.cgi",
                "delivery.cfm", "/doi/pdf/", "/doi/epdf/", "/content/pdf/"]
        for a in soup.find_all("a", href=True):
            h = a["href"].strip()
            if any(p in h.lower() for p in pats): out.append(urljoin(base_url, h))
        seen, res = set(), []
        for u in out:
            if u and u not in seen: seen.add(u); res.append(u)
        return res
    except Exception:
        return []


# =========================================================================== #
# SOURCES 1-4 : XML structuré  (v5-1 : échelle d'identifiants)                #
# =========================================================================== #
def _epmc_search_pmcid(query):
    st, js, *_ = robust_get(
        "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
        params={"query": query, "format": "json", "resultType": "idlist", "pageSize": 5})
    if st != "ok": return ""
    try:
        for h in json.loads(js).get("resultList", {}).get("result", []):
            if h.get("pmcid"):
                return S(h["pmcid"]).replace("PMC", "")
    except Exception:
        pass
    return ""

def _pmid_to_pmcid(pmid):
    params = {"ids": pmid, "format": "json", "email": MAILTO, "idtype": "pmid"}
    if NCBI_KEY: params["api_key"] = NCBI_KEY
    st, js, *_ = robust_get(
        "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/", params=params)
    if st != "ok": return ""
    try:
        for rec in json.loads(js).get("records", []):
            if rec.get("pmcid"):
                return S(rec["pmcid"]).replace("PMC", "")
    except Exception:
        pass
    return ""

def try_epmc(rec):
    """v5 : PMCID direct -> DOI -> PMID -> convertisseur NCBI -> TITRE."""
    pmcid = S(rec.get("oa_pmcid")).replace("PMC", "").strip()
    how = "direct"
    if not pmcid:
        doi = norm_doi(rec.get("oa_doi"))
        if doi:
            pmcid = _epmc_search_pmcid(f'DOI:"{doi}"')
            if pmcid: how = "by_doi"
    if not pmcid:
        pmid = clean_pmid(rec.get("oa_pmid"))
        if pmid:
            pmcid = _epmc_search_pmcid(f"EXT_ID:{pmid} AND SRC:MED")
            if pmcid:
                how = "by_pmid"
            else:
                pmcid = _pmid_to_pmcid(pmid)
                if pmcid: how = "by_idconv"
    if not pmcid:
        title = S(rec.get("oa_title")).strip()
        if len(title) > 25:
            safe = title.replace('"', " ").replace(":", " ")[:200]
            pmcid = _epmc_search_pmcid(f'TITLE:"{safe}"')
            if pmcid: how = "by_title"
    if not pmcid:
        return None, "no_pmcid", None
    st, xml, *_ = robust_get(
        f"https://www.ebi.ac.uk/europepmc/webservices/rest/PMC{pmcid}/fullTextXML")
    if st != "ok": return None, f"epmc_{st}", None
    p = parse_jats(xml)
    return (p, f"epmc_xml_{how}", xml) if p else (None, "epmc_unparsable", None)

def try_pmc(rec):
    pmcid = S(rec.get("oa_pmcid")).replace("PMC", "").strip()
    if not pmcid:                                   # v5-2 : repli PMID
        pmid = clean_pmid(rec.get("oa_pmid"))
        if pmid: pmcid = _pmid_to_pmcid(pmid)
    if not pmcid: return None, "no_pmcid", None
    params = {"db": "pmc", "id": pmcid, "rettype": "xml", "retmode": "xml"}
    if NCBI_KEY: params["api_key"] = NCBI_KEY
    st, xml, *_ = robust_get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi", params=params)
    if st != "ok": return None, f"pmc_{st}", None
    p = parse_jats(xml)
    return (p, "pmc_eutils", xml) if p else (None, "pmc_unparsable", None)

def try_biorxiv(rec):
    doi = norm_doi(rec.get("oa_doi"))
    if not doi: return None, "no_doi", None
    for server in ("biorxiv", "medrxiv"):
        st, js, *_ = robust_get(f"https://api.biorxiv.org/details/{server}/{doi}")
        if st != "ok": continue
        try: coll = json.loads(js).get("collection", [])
        except Exception: coll = []
        if coll:
            st2, xml, *_ = robust_get(f"https://www.{server}.org/content/{doi}.full-text.xml")
            if st2 == "ok":
                p = parse_jats(xml)
                if p: return p, f"{server}_xml", xml
    return None, "not_preprint", None

def try_arxiv(rec):
    doi = norm_doi(rec.get("oa_doi"))
    m = re.search(r"arxiv\.(\d{4}\.\d{4,5})", doi)
    aid = m.group(1) if m else ""
    if not aid: return None, "no_arxiv", None
    if GROBID_URL:
        st, pdf, *_ = robust_get(f"https://arxiv.org/pdf/{aid}.pdf", expect="pdf")
        if st == "ok":
            parsed, how = pdf_to_sections(pdf)
            if parsed: return parsed, f"arxiv_pdf_{how}", pdf
    st, xml, *_ = robust_get("http://export.arxiv.org/api/query",
                             params={"id_list": aid, "max_results": 1})
    if st == "ok":
        try:
            root = etree.fromstring(xml.encode("utf-8"), parser=etree.XMLParser(recover=True))
            summ = root.xpath("//a:entry/a:summary",
                              namespaces={"a": "http://www.w3.org/2005/Atom"})
            if summ:
                return ({"sections": [{"type": "abstract", "title": "abstract",
                                       "text": _txt(summ[0])}], "refs": [], "language": "en"},
                        "arxiv_abstract_only", xml)
        except Exception: pass
    return None, "arxiv_fail", None


# =========================================================================== #
# SOURCE 5-7 : voie web                                                       #
# =========================================================================== #
def try_web(rec, oa_id):
    doi = norm_doi(rec.get("oa_doi"))
    pdf_url = S(rec.get("upw_oa_url"))        # colonne souvent absente -> ""
    landing = ""
    if doi:
        st, js, *_ = robust_get(f"https://api.unpaywall.org/v2/{doi}",
                                params={"email": MAILTO})
        if st == "ok":
            try:
                best = (json.loads(js).get("best_oa_location") or {})
                pdf_url = best.get("url_for_pdf") or pdf_url
                landing = best.get("url") or ""
            except Exception: pass
    if not (pdf_url or landing):
        return None, ("no_oa_url" if doi else "no_doi_for_unpaywall"), None
    for src, url in oa_candidates(pdf_url, landing):
        time.sleep(SLEEP)
        st, body, final, kind = robust_get(rewrite_url(url), expect="auto")
        if st != "ok": continue
        if kind == "pdf":
            parsed, how = pdf_to_sections(body)
            if parsed: return parsed, f"web_{src}_{how}", body
        elif kind == "html":
            for durl in discover_pdf_links(body, final):
                if "pdf" in durl.lower():
                    time.sleep(SLEEP)
                    st2, pb, _f, k2 = robust_get(rewrite_url(durl), expect="pdf")
                    if st2 == "ok" and k2 == "pdf":
                        parsed, how = pdf_to_sections(pb)
                        if parsed: return parsed, f"web_discovered_{how}", pb
            parsed = html_to_sections(body)
            if parsed: return parsed, "web_html", body.encode("utf-8", "ignore")
    return None, "web_failed", None

SOURCES = [try_epmc, try_pmc, try_biorxiv, try_arxiv, try_web]


# =========================================================================== #
# écritures résilientes (v5-5) : local d'abord, synchro Drive par lots        #
# =========================================================================== #
def safe_copy(src, dst, tries=3):
    for a in range(tries):
        try:
            shutil.copy(src, dst); return True
        except OSError:
            time.sleep(2 * (a + 1))
    return False

def sync_to_drive(pending, stats):
    """Copie vers Drive les fichiers écrits en local depuis la dernière synchro."""
    for name in list(pending):
        ok1 = safe_copy(Path(LOCAL_SEC) / name, Path(DRIVE_SEC) / name)
        if not ok1: stats["sync_failed"] += 1
        pending.discard(name)
    for p in list(Path(LOCAL_RAW).glob("*.*")):
        if safe_copy(p, Path(DRIVE_RAW) / p.name):
            try: p.unlink()                       # libère l'espace local
            except Exception: pass
        else:
            stats["sync_failed"] += 1

def has_targets(parsed):
    return _has_targets_shared(parsed)

def source_is_structured(tag):
    tag = S(tag)
    return tag.startswith("epmc_xml") or tag.endswith("_xml") \
           or "grobid" in tag or tag == "pmc_eutils"

def write_outputs(oa_id, rec, parsed, tag, payload, pending):
    if payload is not None:
        if isinstance(payload, (bytes, bytearray)) and payload[:5].startswith(b"%PDF"):
            ext = "pdf"
        elif isinstance(payload, str):
            ext = "xml"
        elif isinstance(payload, (bytes, bytearray)) and (
                b"<html" in payload[:2000].lower()
                or b"<!doctype html" in payload[:2000].lower()):
            ext = "html"
        else:
            ext = "bin"
        mode = "w" if isinstance(payload, str) else "wb"
        try:
            with open(Path(LOCAL_RAW) / f"{oa_id}.{ext}", mode,
                      encoding=None if "b" in mode else "utf-8") as fh:
                fh.write(payload)
        except Exception: pass
    doc = {"oa_id": oa_id, "oa_doi": norm_doi(rec.get("oa_doi")),
           "oa_pmcid": S(rec.get("oa_pmcid")), "source": tag,
           "language": parsed.get("language", ""), "sections": parsed["sections"],
           "refs": parsed["refs"], "has_targets": has_targets(parsed)}
    name = f"{oa_id}.json"
    (Path(LOCAL_SEC) / name).write_text(json.dumps(doc, ensure_ascii=False),
                                        encoding="utf-8")
    pending.add(name)

def process_one(rec, prior_status, pending):
    oa_id = short_id(rec.get("oa_id"))
    if prior_status == "finished":
        return None
    attempts, best = [], None
    for fn in SOURCES:
        try:
            parsed, tag, payload = fn(rec) if fn is not try_web else fn(rec, oa_id)
        except Exception:
            attempts.append(f"{fn.__name__}:exc"); continue
        attempts.append(tag)
        if parsed:
            if prior_status == "reprocess" and not (has_targets(parsed)
                                                    or source_is_structured(tag)):
                best = best or (parsed, tag, payload); continue
            write_outputs(oa_id, rec, parsed, tag, payload, pending)
            return {"oa_id": oa_id, "source": tag, "ok": True,
                    "n_sections": len(parsed["sections"]), "n_refs": len(parsed["refs"]),
                    "has_targets": has_targets(parsed), "attempts": ">".join(attempts)}
    if best and prior_status != "reprocess":
        parsed, tag, payload = best
        write_outputs(oa_id, rec, parsed, tag, payload, pending)
        return {"oa_id": oa_id, "source": tag, "ok": True,
                "n_sections": len(parsed["sections"]), "n_refs": len(parsed["refs"]),
                "has_targets": has_targets(parsed), "attempts": ">".join(attempts)}
    if prior_status == "reprocess":
        return None
    return {"oa_id": oa_id, "source": "", "ok": False, "n_sections": 0, "n_refs": 0,
            "has_targets": False, "attempts": ">".join(attempts)}


# =========================================================================== #
if __name__ == "__main__":
    from collections import Counter
    p = str(IN_FILE).lower()
    df_all = (pd.read_excel(IN_FILE, dtype=str) if p.endswith((".xlsx", ".xls"))
              else pd.read_parquet(IN_FILE))
    df = (df_all[df_all[INCLUDE_COL].astype(str).str.strip().str.lower() == INCLUDE_VAL]
          if INCLUDE_COL in df_all.columns else df_all)
    if RANGE_START or RANGE_END:                       # v5-7
        df = df.iloc[RANGE_START:(RANGE_END if RANGE_END else len(df))]
    print(f"[in] {len(df):,} études à traiter", file=sys.stderr)
    if "oa_pmcid" in df.columns:
        n_pm = df["oa_pmcid"].map(lambda v: bool(S(v).strip())).sum()
        print(f"[in] PMCID pré-remplis : {n_pm:,} ({n_pm/max(len(df),1):.1%})",
              file=sys.stderr)

    # manifeste -> statuts
    status, results = {}, []
    man_src = MANIFEST if Path(MANIFEST).exists() else None
    if man_src:
        try:
            prev = pd.read_parquet(man_src)
            results = prev.to_dict("records")
            for r in results:
                if r.get("ok") and r.get("has_targets"):
                    status[r["oa_id"]] = ("reprocess"
                        if REQUIRE_STRUCTURED_FOR_FINISHED
                        and not source_is_structured(r.get("source")) else "finished")
                elif r.get("ok"):
                    status[r["oa_id"]] = "reprocess"
            n_fin = sum(1 for v in status.values() if v == "finished")
            n_rep = sum(1 for v in status.values() if v == "reprocess")
            print(f"[resume] terminés {n_fin:,} | à améliorer {n_rep:,} | "
                  f"à retenter {len(results) - n_fin - n_rep:,}", file=sys.stderr)
        except Exception as e:
            print(f"[resume] manifeste illisible ({e}) — départ à zéro", file=sys.stderr)
    res_by_id = {r["oa_id"]: r for r in results}

    pending, stats = set(), Counter()
    n, last_saved, t0 = 0, 0, time.time()
    try:
        for _, rec in df.iterrows():
            oid = short_id(rec.get("oa_id"))
            if status.get(oid) == "finished":
                continue
            row = process_one(rec, status.get(oid), pending)
            if row:
                res_by_id[oid] = row; n += 1
            if n and n % SYNC_EVERY == 0 and n != last_saved:
                last_saved = n
                sync_to_drive(pending, stats)                    # v5-5
                mdf = pd.DataFrame(res_by_id.values())
                mdf.to_parquet(LOCAL_MAN, index=False)
                safe_copy(LOCAL_MAN, MANIFEST)
                ok = int(mdf["ok"].sum()); tgt = int(mdf["has_targets"].sum())
                rate = n / max(time.time() - t0, 1)
                print(f"[ft] traités {n:,} | ok {ok:,} | sections cibles {tgt:,} "
                      f"| {rate:.2f}/s", file=sys.stderr)
                if GROBID_URL and not check_grobid(GROBID_URL):
                    print("[warn] GROBID ne répond plus — les PDF suivants seront "
                          "découpés par regex. Redémarrez-le et relancez.", file=sys.stderr)
    except KeyboardInterrupt:
        print("\n[interrompu] synchronisation en cours...", file=sys.stderr)
    finally:
        sync_to_drive(pending, stats)
        man = pd.DataFrame(res_by_id.values())
        man.to_parquet(LOCAL_MAN, index=False)
        safe_copy(LOCAL_MAN, MANIFEST)

    ok = int(man["ok"].sum()); tgt = int(man["has_targets"].sum())
    print(f"\n[done] texte intégral {ok:,}/{len(man):,} ({ok/max(len(man),1):.1%}) ; "
          f"sections cibles {tgt:,} ({tgt/max(len(man),1):.1%})", file=sys.stderr)
    print("[par source]\n" + man[man["ok"]]["source"].value_counts().head(20).to_string(),
          file=sys.stderr)
    fail = man[~man["ok"]]
    if len(fail):
        last = fail["attempts"].astype(str).str.split(">").str[-1]
        print("\n[cause finale des échecs]\n" + last.value_counts().head(10).to_string(),
              file=sys.stderr)
    if stats["sync_failed"]:
        print(f"\n[!] {stats['sync_failed']:,} synchronisations Drive échouées — "
              f"les fichiers sont dans {LOCAL_SEC}", file=sys.stderr)